# 01 · Your first Kelvo query

Register an open CSV, run SQL, inspect typed Arrow results, and validate the result against the original file.

Open this notebook in Google Colab with a CPU runtime, or use a Linux x86-64
Jupyter environment with Python 3.11 or newer. No database account is required.
Run the cells in order. The first cell downloads a verified Kelvo preview binary
and installs pinned Python readers; it does not compile Kelvo.

Every analytical SQL query below runs through the **Kelvo executable**. PyArrow
reads its Arrow output, and pandas/matplotlib format small results. The pinned
DuckDB driver materializes execution before delivering Arrow batches; delivery
backpressure is not a worker memory bound.


**Data:** [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/),
[CC0 license](https://github.com/allisonhorst/palmerpenguins/blob/8957207b78d6ccd1b4654a9dd9c9041b657478ab/DESCRIPTION).
Citation: Horst, Hill and Gorman (2020), *palmerpenguins: Palmer Archipelago
(Antarctica) penguin data*, [DOI: 10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218).
The helper downloads the pinned original CSV and verifies its SHA-256.


[All notebooks](https://github.com/SYNEHQ/kelvo-go/blob/cargo/notebooks/README.md)
| [Kelvo documentation](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/README.md)

## Set up a private lab


In [ ]:
# Default setup installs pinned Python packages and verifies the reviewed helper.
# Operators validating a checkout can explicitly set KELVO_NOTEBOOK_SKIP_INSTALL=1
# and KELVO_NOTEBOOK_HELPER=/absolute/path/to/kelvo_notebooks.py before starting.
import hashlib
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request

KELVO_NOTEBOOK_REF = "eacb7e2aedd8020575ecfede53db819112c40199"
KELVO_HELPER_SHA256 = "86696a66a3c903eb8f18d529fcef055e8bee72234da324b8776529138436d9a9"
if os.environ.get("KELVO_NOTEBOOK_SKIP_INSTALL") != "1":
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet",
        "pyarrow==25.0.1", "pandas==2.3.3", "matplotlib==3.10.7", "PyYAML==6.0.3",
    ], check=True)

helper_directory = None
local_helper = os.environ.get("KELVO_NOTEBOOK_HELPER")
if local_helper:
    helper_path = Path(local_helper).expanduser().resolve(strict=True)
else:
    helper_url = (
        "https://raw.githubusercontent.com/SYNEHQ/kelvo-go/"
        + KELVO_NOTEBOOK_REF + "/notebooks/kelvo_notebooks.py"
    )
    request = urllib.request.Request(helper_url, headers={"User-Agent": "kelvo-notebooks"})
    with urllib.request.urlopen(request, timeout=60) as response:
        final_url = response.geturl()
        if urllib.parse.urlsplit(final_url).scheme != "https" or final_url != helper_url:
            raise ValueError("Helper download did not retain its pinned HTTPS URL")
        payload = response.read(1024 * 1024 + 1)
    if len(payload) > 1024 * 1024:
        raise ValueError("Unexpected helper size")
    if hashlib.sha256(payload).hexdigest() != KELVO_HELPER_SHA256:
        raise ValueError("Helper checksum mismatch")
    helper_directory = Path(tempfile.mkdtemp(prefix="kelvo-notebook-helper-"))
    helper_path = helper_directory / "kelvo_notebooks.py"
    helper_path.write_bytes(payload)

spec = importlib.util.spec_from_file_location("kelvo_notebooks", helper_path)
helper = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helper
spec.loader.exec_module(helper)
from kelvo_notebooks import Lab, remote_query

lab = Lab.create()
print("Kelvo workspace:", lab.root)


## Register the source

A YAML catalog names the files that a query is allowed to use. SQL refers to the
registered alias, not an arbitrary filesystem path.


In [ ]:
penguins = lab.penguins()
config = lab.catalog(sources=[{"id": "penguins", "type": "csv", "path": str(penguins)}])
print(config.read_text())


## Ask a question

How many observations are available for each species? Only this three-row
summary is converted to a pandas DataFrame.


In [ ]:
result = lab.query(config, """
    SELECT species, COUNT(*)::BIGINT AS observations
    FROM penguins GROUP BY species ORDER BY species
""", sources=["penguins"])
table = result.table()
display(table.to_pandas())
print(table.schema)
print(result.stats)


## Verify the answer

The CSV reader below is an independent check of the grouping, not an alternative
query engine. These assertions must pass before using the result.


In [ ]:
import csv
from collections import Counter
with penguins.open(newline="") as stream:
    expected = Counter(row["species"] for row in csv.DictReader(stream))
actual = dict(zip(table["species"].to_pylist(), table["observations"].to_pylist()))
assert actual == dict(expected)
assert sum(actual.values()) == 344
assert result.stats["rows"] == 3
assert result.stats["backend"] == "duckdb"
assert result.stats["engine_streaming"] is False
print("Verified species counts:", actual)


## Where Kelvo fits

The same application can consume Arrow from a local CLI export or an authenticated
HTTP gateway. Registered connectors and accelerated snapshots sit behind the
query boundary. Continue with [parameters and types](https://github.com/SYNEHQ/kelvo-go/blob/cargo/notebooks/02-parameters-nulls-and-types.ipynb)
or the [HTTP gateway lesson](https://github.com/SYNEHQ/kelvo-go/blob/cargo/notebooks/15-authenticated-http-gateway.ipynb).


## Clean up

Run this cell after inspecting or saving your results. Re-run the setup and data
cells if you want to restart the lesson after cleanup.


In [ ]:
# Close local processes and delete this lesson's private temporary workspace.
# Downloaded release/data caches, if configured by the helper, remain reusable.
lab.close()
if helper_directory is not None:
    shutil.rmtree(helper_directory, ignore_errors=True)
